# Trabajo Práctico 2 - Grupo 02

### Modelo XGBoost

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen


El objetivo es entrenar un modelo de XGBoost sobre dos representaciones de texto (TF-IDF y BoW) con búsqueda de hiperparámetros, comparando su desempeño contra el baseline de Naive Bayes (~ 0.66 F1-macro en Kaggle) y contra el de Random Forest (~ 0.65 F1-macro en Kaggle).

XGBoost (eXtreme Gradient Boosting) es una técnica de aprendizaje por ensambles (ensemble learning), específicamente de tipo boosting. Combina múltiples modelos predictivos básicos (conocidos como "aprendices débiles"), generalmente árboles de decisión, para crear un modelo final altamente preciso y robusto.

## Importación e instalación de dependencias

In [1]:
!pip install click
!pip install -q spacy
!python -m spacy download es_core_news_sm
!pip install numpy pandas matplotlib seaborn xgboost scikit-learn
!pip install nltk

     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     ------------ --------------------------- 3.9/12.9 MB 55.8 MB/s eta 0:00:01
     ---------------------------------------- 12.9/12.9 MB 60.8 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight

In [3]:
import nltk
nltk.download("sentiwordnet")
nltk.download("wordnet")
nltk.download("omw-1.4")
from scipy.sparse import hstack, csr_matrix

[nltk_data] Downloading package sentiwordnet to
[nltk_data]     C:\Users\gonza\AppData\Roaming\nltk_data...
[nltk_data]   Package sentiwordnet is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gonza\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\gonza\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [8]:
import sys
sys.path.insert(0, "../../../..")  # sube cuatro niveles: v7 → XGBoost → Entregas → TP2

In [9]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED, extract_features, VectorizerPlusFeaturesPipeline
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [11]:
train = pd.read_csv("../../../../data/train.csv")
test  = pd.read_csv("../../../../data/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [12]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N8: XGBoost + BoW con feature engineering pero sin optimización de hiperparámetros

In [13]:
pipe = VectorizerPlusFeaturesPipeline(
    vectorizer=make_bow(),
    classifier=XGBClassifier(n_estimators=200, random_state=SEED, n_jobs=-1, eval_metric="mlogloss")
)

weights_train = compute_sample_weight("balanced", y_train)
pipe.fit(X_train, X_train_raw, y_train, sample_weight=weights_train)
y_pred = pipe.predict(X_val, X_val_raw)
evaluate("xgb_bow_features_v7_2", y_val, y_pred, 
         hyperparams={"n_estimators": 200, "vectorizer": "BoW", "features": True, "class_weight": "balanced"})

# Reentrenar en train completo y generar submission
y_full = np.concatenate([y_train, y_val])
weights_full = compute_sample_weight("balanced", y_full)
pipe.fit(np.concatenate([X_train, X_val]), 
         np.concatenate([X_train_raw, X_val_raw]), y_full, sample_weight=weights_full)

save_model(pipe, "xgb_bow_features_v7_2")

make_submission(pipe, pd.DataFrame({"id": test["id"], "text": X_test}), 
                "submission_xgb_bow_features_v7_2.csv", 
                raw_texts=test["text"].values);


=== xgb_bow_features_v7_2 ===
Hiperparámetros: {'n_estimators': 200, 'vectorizer': 'BoW', 'features': True, 'class_weight': 'balanced'}

F1-macro:  0.6512
Precision: 0.6535
Recall:    0.6530
Accuracy:  0.6884

              precision    recall  f1-score   support

    negativa     0.7754    0.7218    0.7477      4080
      neutra     0.3862    0.4760    0.4264      2040
    positiva     0.7989    0.7613    0.7796      4080

    accuracy                         0.6884     10200
   macro avg     0.6535    0.6530    0.6512     10200
weighted avg     0.7070    0.6884    0.6962     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      2945     853       282
neutra         569     971       500
positiva       284     690      3106
Modelo guardado → models\xgb_bow_features_v7_2.joblib
Predicción guardada → submissions\submission_xgb_bow_features_v7_2.csv  (8500 predicciones)
Distribución: clase 0: 37.4%, clase 1: 24.8%, clase 2: 37.8%
